# OneVoice V2 — streaming semantics và reliability

Notebook này chỉ điều phối Colab/Drive. Runtime thật nằm trong `src/pipeline.py`: WAV được phát lại thành frame 32 ms, đi qua denoiser/VAD/ASR rolling/semantic commit/MT/TTS và ghi trace cùng latency. Không phát ra loa trong chế độ replay.

Mặc định dùng safety WAV đã có checksum để smoke test nhanh; đổi `DIRECTION` và `INPUT_FILE` nếu muốn chạy một WAV khác. Báo cáo được ghi trên Drive và có thể chạy lại.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path
import json, os, subprocess, sys
GITHUB_REPO = 'https://github.com/Platypus27-coder/OneVoice.git'
BRANCH = 'main'
REPO = Path('/content/OneVoice')
DRIVE_ROOT = Path('/content/drive/MyDrive/OneVoice')
if (REPO / '.git').is_dir():
    subprocess.run(['git', '-C', str(REPO), 'pull', '--ff-only', 'origin', BRANCH], check=True)
else:
    subprocess.run(['git', 'clone', '--depth', '1', '--branch', BRANCH, GITHUB_REPO, str(REPO)], check=True)
os.chdir(REPO)
os.environ['PYTHONUNBUFFERED'] = '1'
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'soundfile', 'PyYAML'], check=True)
CONFIG = DRIVE_ROOT / 'configs/runtime_demo_local.yaml'
DIRECTION = 'vi2en'  # 'vi2en' hoặc 'en2vi'
REPORT_DIR = DRIVE_ROOT / 'reports/streaming_v2_smoke' / DIRECTION
DEFAULT_INPUT = DRIVE_ROOT / 'artifacts/safety_audio_v1' / (
    'SAFE2_0001_en2vi.wav' if DIRECTION == 'vi2en' else 'SAFE2_0001_vi2en.wav'
)
INPUT_FILE = DEFAULT_INPUT  # thay bằng Path('/content/drive/MyDrive/.../file.wav') nếu cần
if not CONFIG.is_file(): raise FileNotFoundError(f'Missing runtime config: {CONFIG}')
if not INPUT_FILE.is_file(): raise FileNotFoundError(f'Missing input WAV: {INPUT_FILE}')
print('Source:', REPO)
print('Direction:', DIRECTION, '| Input:', INPUT_FILE, '| Reports:', REPORT_DIR)


In [ ]:
command = [
    sys.executable, 'src/pipeline.py',
    '--config', str(CONFIG), '--direction', DIRECTION,
    '--profile', 'development', '--offline',
    '--stream-file', str(INPUT_FILE), '--report-dir', str(REPORT_DIR),
]
print('> ' + ' '.join(map(str, command)), flush=True)
process = subprocess.Popen(command, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1, env={**os.environ, 'PYTHONUNBUFFERED': '1'})
assert process.stdout is not None
for line in process.stdout:
    print(line, end='', flush=True)
code = process.wait()
if code:
    raise RuntimeError(f'Streaming smoke failed with exit code {code}; inspect the full log above and stream_result.json on Drive.')
result_path = REPORT_DIR / 'stream_result.json'
result = json.loads(result_path.read_text(encoding='utf-8'))
print(json.dumps({k: result[k] for k in ('direction', 'frame_samples', 'frame_ms', 'frames_submitted', 'commits', 'dropped_audio_frames', 'fatal_error')}, ensure_ascii=False, indent=2))


In [ ]:
result = json.loads((REPORT_DIR / 'stream_result.json').read_text(encoding='utf-8'))
latency = json.loads((REPORT_DIR / 'latency_summary.json').read_text(encoding='utf-8')) if (REPORT_DIR / 'latency_summary.json').is_file() else {}
print('Commits:', result['commits'], '| commit IDs:', result['commit_ids'])
print('Latency summary:', json.dumps(latency, ensure_ascii=False, indent=2))
print('Hypothesis events:', len(result['hypothesis_trace']), '| chunks:', len(result['chunks']))
print('Report directory:', REPORT_DIR)
